# tribe-bench: Setup & Smoke Test (the gate)

Run on **Kaggle** with **GPU T4 x2** and **internet ON** (Settings panel).

This one notebook answers the three questions everything else waits on:

1. **Does it install?** (G016 — deps are public wheels; `neuralset` needs Python >=3.12)
2. **Does it fit a T4?** (G005 — peak VRAM during one prediction)
3. **Does modality ablation actually work?** (G018 — BrainLens's whole mechanic)

If step 8 shows identical output for full vs audio-removed, BrainLens is dead as designed
and we pivot to NeuroCheck-only. Run cells top to bottom. ~15-30 min incl. downloads.

## 1. Install

Kaggle is usually on Python 3.11; `neuralset` wants >=3.12. If the pins below fail on the
Python version, that is the thing to flag — note it and stop.

In [ ]:
import sys
print('Python:', sys.version)

# TRIBE v2 is a GitHub repo, not a PyPI package — clone and install editable.
!git clone --depth 1 https://github.com/facebookresearch/tribev2.git /kaggle/working/tribev2 || echo 'already cloned'
!pip install -q -e /kaggle/working/tribev2

# Meta's support libs (public wheels, G016 resolved). Pin the versions we verified.
!pip install -q 'neuralset==0.0.2' 'neuraltrain==0.0.2' exca

# tribe-bench itself (private repo — needs a token if not already cloned locally).
!git clone --depth 1 https://github.com/codesbydevesh/tribe-bench.git /kaggle/working/tribe-bench || echo 'already cloned'
!pip install -q -e /kaggle/working/tribe-bench
print('\ninstall step done — check above for any resolver/version errors')

## 2. Environment check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
else:
    print('NO GPU — turn on the T4 x2 accelerator in Kaggle Settings, then re-run.')

## 3. HuggingFace login (required)

TRIBE v2 pulls **LLaMA 3.2-3B**, which is gated. Request access once at
huggingface.co/meta-llama/Llama-3.2-3B, then paste a read token below (or add it as a
Kaggle secret named `HF_TOKEN`).

In [ ]:
import os
from huggingface_hub import login
token = os.environ.get('HF_TOKEN', '')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if token:
    login(token=token)
    print('HF login OK')
else:
    print('No HF_TOKEN found. Set a Kaggle secret HF_TOKEN or run: login(token="hf_...")')

## 4. Verify imports + the claims DB

In [ ]:
from tribe_tools import model, inference, atlas, viz, cache, video_utils
from tribe_tools.model import load_model, predict_single, MODALITY_MASKS, _find_features_to_use
from neurocheck.claims import load_claims, validate_claims

claims = load_claims()
errors = validate_claims()
print(f'Claims: {len(claims)}, validation errors: {len(errors)}')
print('tribe-bench imports: OK')

## 5. Make a self-contained test clip

A 10 s clip with a moving visual pattern **and** a 440 Hz tone, so both a video and an
audio stream exist — needed to tell whether masking audio actually changes the output.

In [ ]:
from pathlib import Path
video_path = Path('/kaggle/working/test_clip.mp4')
!ffmpeg -y -f lavfi -i testsrc=duration=10:size=320x240:rate=10 -f lavfi -i sine=frequency=440:duration=10 -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest {video_path} -loglevel error
print('clip exists:', video_path.exists(), '| size:', video_path.stat().st_size if video_path.exists() else 0, 'bytes')

## 6. Load the model (times it)

In [ ]:
import time
t0 = time.time()
tribe = load_model(device='cuda', cache_folder=Path('/kaggle/working/cache'))
load_time = time.time() - t0
print(f'Model loaded in {load_time:.1f}s')

## 7. Full prediction + peak VRAM (does it fit a T4? — G005)

If this runs without OOM, TRIBE fits free-tier hardware. Peak VRAM is the number to record.

In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
preds, segments = predict_single(tribe, video_path)  # full, all modalities
infer_time = time.time() - t0
peak_vram = torch.cuda.max_memory_allocated() / 1e9
print('=== FULL PREDICTION ===')
print('shape:', preds.shape, '| dtype:', preds.dtype)
print(f'range: [{preds.min():.4f}, {preds.max():.4f}]  mean: {preds.mean():.4f}')
print('segments kept:', len(segments))
print(f'inference time: {infer_time:.1f}s')
print(f'PEAK VRAM: {peak_vram:.2f} GB  (T4 usable ~15 GB)')

## 8. THE decisive test — does modality ablation work? (G018)

First show where `features_to_use` actually lives, then run a video-only pass (audio + text
masked) and compare it to the full pass. **They must differ.** If `predict_single` raises the
G018 error, or the arrays are identical, BrainLens's core mechanic does not work as built
→ pivot to NeuroCheck-only.

In [ ]:
import numpy as np
print('dir(model):', [a for a in dir(tribe) if not a.startswith('__')][:40])
print('has .data:', hasattr(tribe, 'data'), '| has .xp:', hasattr(tribe, 'xp'))
if hasattr(tribe, 'data'):
    print('dir(model.data):', [a for a in dir(tribe.data) if not a.startswith('__')][:40])
loc = _find_features_to_use(tribe)
print('\n_find_features_to_use ->', loc if loc is None else (type(loc[0]).__name__, loc[1], getattr(loc[0], loc[1])))

In [ ]:
verdict = None
try:
    preds_vid, _ = predict_single(tribe, video_path, features_to_mask=['audio', 'text'])
    same_shape = preds_vid.shape == preds.shape
    if same_shape:
        identical = np.allclose(preds_vid, preds)
        maxdiff = float(np.abs(preds_vid - preds).max())
        verdict = 'DEAD (identical output)' if identical else f'WORKS (max abs diff {maxdiff:.4f})'
    else:
        verdict = f'WORKS (shape changed: full {preds.shape} vs video-only {preds_vid.shape})'
except RuntimeError as e:
    verdict = f'PATH NOT FOUND (G018): {e}'
except Exception as e:
    verdict = f'ERROR: {type(e).__name__}: {e}'
print('ABLATION VERDICT:', verdict)

## 9. Save + record

Save outputs to `/kaggle/working` (download before the session ends), then paste the block
into `ops/source-of-truth.md` and update G005 / G018 in `ops/knowledge-gaps.md`.

In [ ]:
import numpy as np
np.save('/kaggle/working/smoke_full.npy', preds)
print('=== RECORD IN ops/source-of-truth.md ===')
print(f'output shape        : {preds.shape}')
print(f'n_vertices          : {preds.shape[1]}')
print(f'peak VRAM (GB)      : {peak_vram:.2f}   -> G005')
print(f'load time (s)       : {load_time:.1f}')
print(f'inference time (s)  : {infer_time:.1f}')
print(f'ablation (G018)     : {verdict}')
print(f'python              : {sys.version.split()[0]}')
print('=========================================')